# 🏦 Financial Agent Evaluation Suite

Evaluate your Azure AI Foundry agents against a structured financial test pack.

**Agents under test:**
- `agent-summarizer` — Summarizes financial documents
- `User-clarification-agent` — Asks follow-up questions
- `report-generator-agent` — Generates structured financial reports

**Test data:** 24 Q&A pairs across 3 synthetic financial documents

---
## Section 0 — Configuration

In [ ]:
import os

# ── Your Foundry config ──────────────────────────────────────────────
ENDPOINT   = os.environ["AZURE_AI_ENDPOINT"]
MODEL      = "gpt-4o"   # deployment used for routing / scoring

# Agent names exactly as they appear in Foundry
AGENTS = {
    "summarizer":    "agent-summarizer",
    "clarification": "User-clarification-agent",
    "reporter":      "report-generator-agent",
}

# Path to the extracted test pack
TEST_PACK_DIR = "_test_pack"

print("✅ Config set")

---
## Section 1 — Imports & Clients

In [ ]:
import json, os, time, textwrap, pathlib
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML, Markdown

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=ENDPOINT, credential=credential)
oai = project_client.get_openai_client()

print("✅ Clients ready")

---
## Section 2 — Load Test Pack

In [ ]:
# Load evaluation dataset
eval_df = pd.read_csv(os.path.join(TEST_PACK_DIR, "financial_eval_dataset.csv"))

# Preview
display(HTML(f"<b>{len(eval_df)} test cases</b> across <b>{eval_df['document_id'].nunique()}</b> documents"))

doc_summary = eval_df.groupby(["document_id", "document_file"]).size().reset_index(name="questions")

html = "<table style='border-collapse:collapse; margin:10px 0;'>"
html += "<tr style='background:#1a1a2e; color:white;'><th style='padding:8px 16px;'>Doc ID</th><th style='padding:8px 16px;'>File</th><th style='padding:8px 16px;'>Questions</th></tr>"
for _, r in doc_summary.iterrows():
    html += f"<tr style='border-bottom:1px solid #ddd;'><td style='padding:6px 16px;'>{r['document_id']}</td><td style='padding:6px 16px;'><code>{r['document_file']}</code></td><td style='padding:6px 16px; text-align:center;'>{r['questions']}</td></tr>"
html += "</table>"
display(HTML(html))

---
## Section 3 — Helper Functions

In [ ]:
def call_agent(agent_name: str, prompt: str) -> str:
    """Invoke a Foundry prompt agent by name and return the response text."""
    resp = oai.responses.create(
        extra_body={
            "agent_reference": {
                "name": agent_name,
                "type": "agent_reference",
            }
        },
        input=prompt,
    )
    return resp.output_text


def score_answer(question: str, expected: str, actual: str) -> dict:
    """Use the LLM to score how well the actual answer matches expected."""
    scoring_prompt = f"""You are a strict evaluator. Compare the ACTUAL answer to the EXPECTED answer for the given question.

Question: {question}
Expected: {expected}
Actual: {actual}

Respond ONLY with valid JSON (no markdown, no code fences):
{{{{
  "match": true/false,
  "score": 0-100,
  "reason": "brief explanation"
}}}}

Rules:
- match=true if the key facts are present (exact wording not required)
- score 90-100: all key facts present
- score 60-89: most key facts present, minor omissions
- score 30-59: partial match, significant omissions
- score 0-29: wrong or missing"""

    resp = oai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": scoring_prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"match": False, "score": 0, "reason": f"Scoring parse error: {raw[:120]}"}


def progress_bar(current, total, label=""):
    """Display an HTML progress bar."""
    pct = int(current / total * 100)
    color = "#4CAF50" if pct == 100 else "#2196F3"
    display(HTML(f"""
    <div style='margin:4px 0;'>
        <span style='font-size:13px;'>{label} {current}/{total}</span>
        <div style='background:#eee; border-radius:4px; overflow:hidden; height:20px; width:400px; display:inline-block; vertical-align:middle;'>
            <div style='background:{color}; height:100%; width:{pct}%; transition:width 0.3s;'></div>
        </div>
    </div>"""))

print("✅ Helpers defined")

---
## Section 4 — Agent Connectivity Check
Quick smoke test: send one question to each agent.

In [ ]:
connectivity_results = {}

for label, agent_name in AGENTS.items():
    try:
        t0 = time.time()
        reply = call_agent(agent_name, "Say OK in one word.")
        latency = time.time() - t0
        ok = len(reply.strip()) > 0
        connectivity_results[label] = {"status": "✅" if ok else "⚠️", "latency": f"{latency:.1f}s", "preview": reply.strip()[:80]}
    except Exception as e:
        connectivity_results[label] = {"status": "❌", "latency": "-", "preview": str(e)[:80]}

# Render
html = "<table style='border-collapse:collapse; margin:10px 0; width:100%;'>"
html += "<tr style='background:#1a1a2e; color:white;'><th style='padding:8px 12px;'>Agent</th><th style='padding:8px 12px;'>Status</th><th style='padding:8px 12px;'>Latency</th><th style='padding:8px 12px;'>Response Preview</th></tr>"
for label, r in connectivity_results.items():
    bg = '#e8f5e9' if r['status'] == '✅' else '#ffebee'
    html += f"<tr style='background:{bg}; border-bottom:1px solid #ddd;'>"
    html += f"<td style='padding:6px 12px;'><b>{AGENTS[label]}</b></td>"
    html += f"<td style='padding:6px 12px; text-align:center; font-size:18px;'>{r['status']}</td>"
    html += f"<td style='padding:6px 12px; text-align:center;'>{r['latency']}</td>"
    html += f"<td style='padding:6px 12px;'><code>{r['preview']}</code></td></tr>"
html += "</table>"
display(HTML(html))

---
## Section 5 — Document Summarization Test
Feed each financial document's content description to the summarizer agent and then score the summary against the eval questions.

In [ ]:
# Build per-document context from the eval dataset (evidence column)
doc_contexts = {}
for doc_id, group in eval_df.groupby("document_id"):
    file_name = group.iloc[0]["document_file"]
    evidence_lines = group["evidence"].tolist()
    doc_contexts[doc_id] = {
        "file": file_name,
        "context": "\n".join(f"- {e}" for e in evidence_lines),
    }

# Get summaries
summaries = {}
for doc_id, info in doc_contexts.items():
    print(f"📄 Summarizing {info['file']}...")
    prompt = f"""Summarize this financial document. Include: company name, reporting period, key financial figures (revenue, net income, operating expenses, operating margin), risks, and management outlook.

Document: {info['file']}
Content:
{info['context']}"""
    summaries[doc_id] = call_agent(AGENTS["summarizer"], prompt)
    print(f"  ✅ Got {len(summaries[doc_id])} chars")

print(f"\n🎯 {len(summaries)} documents summarized")

### View Summaries

In [ ]:
for doc_id, summary in summaries.items():
    file_name = doc_contexts[doc_id]["file"]
    display(HTML(f"""
    <div style='border:1px solid #ccc; border-radius:8px; padding:16px; margin:12px 0; background:#f9f9f9;'>
        <h3 style='margin-top:0;'>📄 {file_name} <span style='color:#888; font-size:14px;'>({doc_id})</span></h3>
        <pre style='white-space:pre-wrap; font-size:13px; line-height:1.5;'>{summary}</pre>
    </div>"""))

---
## Section 6 — Q&A Accuracy Evaluation
Ask the summarizer each question (with the document context) and score against expected answers.

In [ ]:
results = []
total = len(eval_df)

for idx, row in eval_df.iterrows():
    doc_id = row["document_id"]
    summary = summaries.get(doc_id, "")

    qa_prompt = f"""Based on this financial summary, answer the question concisely.

Summary:
{summary}

Question: {row['question']}
Answer in one or two sentences."""

    try:
        actual = call_agent(AGENTS["summarizer"], qa_prompt)
    except Exception as e:
        actual = f"ERROR: {e}"

    score_result = score_answer(row["question"], row["expected_answer"], actual)

    results.append({
        "sample_id": row["sample_id"],
        "document_id": doc_id,
        "document_file": row["document_file"],
        "question": row["question"],
        "expected": row["expected_answer"],
        "actual": actual.strip()[:300],
        "match": score_result.get("match", False),
        "score": score_result.get("score", 0),
        "reason": score_result.get("reason", ""),
    })

    if (idx + 1) % 4 == 0 or idx == total - 1:
        from IPython.display import clear_output
        clear_output(wait=True)
        progress_bar(idx + 1, total, "Evaluating Q&A:")

results_df = pd.DataFrame(results)
print(f"\n✅ Evaluation complete — {len(results_df)} questions scored")

---
## Section 7 — Results Dashboard

In [ ]:
# ── Overall metrics ──
avg_score = results_df["score"].mean()
match_rate = results_df["match"].mean() * 100
pass_count = results_df["match"].sum()
fail_count = len(results_df) - pass_count

# Color based on score
def score_color(s):
    if s >= 90: return "#4CAF50"  # green
    if s >= 60: return "#FF9800"  # orange
    return "#f44336"              # red

# ── KPI cards ──
display(HTML(f"""
<div style='display:flex; gap:16px; margin:16px 0;'>
  <div style='flex:1; background:{score_color(avg_score)}; color:white; border-radius:12px; padding:20px; text-align:center;'>
    <div style='font-size:36px; font-weight:bold;'>{avg_score:.0f}</div>
    <div style='font-size:14px; opacity:0.9;'>Avg Score</div>
  </div>
  <div style='flex:1; background:{score_color(match_rate)}; color:white; border-radius:12px; padding:20px; text-align:center;'>
    <div style='font-size:36px; font-weight:bold;'>{match_rate:.0f}%</div>
    <div style='font-size:14px; opacity:0.9;'>Match Rate</div>
  </div>
  <div style='flex:1; background:#4CAF50; color:white; border-radius:12px; padding:20px; text-align:center;'>
    <div style='font-size:36px; font-weight:bold;'>{pass_count}</div>
    <div style='font-size:14px; opacity:0.9;'>Passed</div>
  </div>
  <div style='flex:1; background:{"#f44336" if fail_count > 0 else "#4CAF50"}; color:white; border-radius:12px; padding:20px; text-align:center;'>
    <div style='font-size:36px; font-weight:bold;'>{fail_count}</div>
    <div style='font-size:14px; opacity:0.9;'>Failed</div>
  </div>
</div>
"""))

In [ ]:
# ── Score by Document (bar chart) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Avg score per document
doc_scores = results_df.groupby("document_file")["score"].mean()
colors = [score_color(s) for s in doc_scores.values]
bars = axes[0].barh(doc_scores.index, doc_scores.values, color=colors, edgecolor="white", height=0.5)
axes[0].set_xlim(0, 105)
axes[0].set_xlabel("Average Score")
axes[0].set_title("📊 Score by Document", fontsize=14, fontweight="bold")
for bar, val in zip(bars, doc_scores.values):
    axes[0].text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.0f}", va="center", fontweight="bold")
axes[0].invert_yaxis()

# Chart 2: Pass/Fail per document
doc_pass = results_df.groupby("document_file")["match"].sum()
doc_fail = results_df.groupby("document_file")["match"].count() - doc_pass
x = range(len(doc_pass))
axes[1].bar(x, doc_pass.values, color="#4CAF50", label="Pass", width=0.5)
axes[1].bar(x, doc_fail.values, bottom=doc_pass.values, color="#f44336", label="Fail", width=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f.split("_")[0] for f in doc_pass.index], rotation=15)
axes[1].set_ylabel("Questions")
axes[1].set_title("✅ Pass / Fail by Document", fontsize=14, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("eval_results_by_document.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: eval_results_by_document.png")

In [ ]:
# ── Score Distribution (histogram + heatmap) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(results_df["score"], bins=[0,30,60,90,100], color=["#f44336"],
             edgecolor="white", rwidth=0.85)
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("📈 Score Distribution", fontsize=14, fontweight="bold")
axes[0].set_xticks([15, 45, 75, 95])
axes[0].set_xticklabels(["0-29\nWrong", "30-59\nPartial", "60-89\nMostly", "90-100\nFull"])

# Heatmap: question type × document
# Extract question category
def categorize_q(q):
    q_lower = q.lower()
    if "company name" in q_lower: return "Company"
    if "period" in q_lower: return "Period"
    if "revenue" in q_lower: return "Revenue"
    if "net income" in q_lower: return "Net Income"
    if "operating expense" in q_lower: return "OpEx"
    if "margin" in q_lower: return "Margin"
    if "risk" in q_lower: return "Risks"
    if "guidance" in q_lower or "outlook" in q_lower or "recommend" in q_lower: return "Outlook"
    return "Other"

results_df["category"] = results_df["question"].apply(categorize_q)
pivot = results_df.pivot_table(index="category", columns="document_id", values="score", aggfunc="mean")

im = axes[1].imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
axes[1].set_xticks(range(len(pivot.columns)))
axes[1].set_xticklabels(pivot.columns)
axes[1].set_yticks(range(len(pivot.index)))
axes[1].set_yticklabels(pivot.index)
axes[1].set_title("🗺️ Score Heatmap", fontsize=14, fontweight="bold")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not pd.isna(val):
            axes[1].text(j, i, f"{val:.0f}", ha="center", va="center", fontweight="bold",
                        color="white" if val < 50 else "black")

plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.savefig("eval_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: eval_score_distribution.png")

---
## Section 8 — Detailed Results Table

In [ ]:
# ── Color-coded detailed results ──
html = """
<style>
  .eval-table { border-collapse:collapse; width:100%; font-size:13px; }
  .eval-table th { background:#1a1a2e; color:white; padding:8px 10px; text-align:left; position:sticky; top:0; }
  .eval-table td { padding:6px 10px; border-bottom:1px solid #eee; vertical-align:top; }
  .eval-table tr:hover { background:#f5f5f5; }
  .score-badge { display:inline-block; padding:2px 10px; border-radius:12px; color:white; font-weight:bold; font-size:12px; }
</style>
<div style='max-height:600px; overflow-y:auto; border:1px solid #ddd; border-radius:8px;'>
<table class='eval-table'>
<tr><th>#</th><th>Doc</th><th>Question</th><th>Expected</th><th>Actual</th><th>Score</th><th>Reason</th></tr>
"""

for _, r in results_df.iterrows():
    sc = r["score"]
    bg = score_color(sc)
    icon = "✅" if r["match"] else "❌"
    html += f"""<tr>
        <td>{r['sample_id']}</td>
        <td><code>{r['document_id']}</code></td>
        <td>{r['question']}</td>
        <td style='max-width:200px;'>{r['expected']}</td>
        <td style='max-width:250px;'>{r['actual'][:150]}</td>
        <td><span class='score-badge' style='background:{bg};'>{icon} {sc}</span></td>
        <td style='font-size:12px; color:#666;'>{r['reason'][:100]}</td>
    </tr>"""

html += "</table></div>"
display(HTML(html))

---
## Section 9 — Clarification Agent Test
Feed summaries to the clarification agent and verify it asks relevant follow-up questions.

In [ ]:
clarification_results = []

for doc_id, summary in summaries.items():
    file_name = doc_contexts[doc_id]["file"]
    print(f"🔍 Testing clarification agent on {file_name}...")

    prompt = f"""I received this financial summary. What follow-up questions would you ask to better understand the company's position?

Summary:
{summary}"""

    try:
        reply = call_agent(AGENTS["clarification"], prompt)
        # Count questions (lines ending with ?)
        q_count = reply.count("?")
        clarification_results.append({
            "document": file_name,
            "questions_asked": q_count,
            "response": reply.strip(),
            "status": "✅" if q_count >= 2 else "⚠️",
        })
    except Exception as e:
        clarification_results.append({"document": file_name, "questions_asked": 0, "response": str(e), "status": "❌"})

# Display
for cr in clarification_results:
    display(HTML(f"""
    <div style='border-left:4px solid {"#4CAF50" if cr["status"]=="✅" else "#FF9800"}; padding:12px 16px; margin:10px 0; background:#fafafa; border-radius:0 8px 8px 0;'>
        <b>{cr['status']} {cr['document']}</b> — {cr['questions_asked']} follow-up question(s)
        <details style='margin-top:8px;'>
            <summary style='cursor:pointer; color:#1976D2;'>Show response</summary>
            <pre style='white-space:pre-wrap; font-size:12px; margin-top:8px;'>{cr['response']}</pre>
        </details>
    </div>"""))

---
## Section 10 — Report Generator Test
Ask the report generator to produce a consolidated report from all summaries.

In [ ]:
all_summaries = "\n\n---\n\n".join(
    f"Document: {doc_contexts[did]['file']}\n{s}" for did, s in summaries.items()
)

report_prompt = f"""Generate a consolidated financial report covering all three companies below. Include:
1. Executive Summary
2. Company-by-company highlights (revenue, net income, margin)
3. Common risks and themes
4. Recommendations

{all_summaries}"""

print("📝 Generating consolidated report...")
report = call_agent(AGENTS["reporter"], report_prompt)

display(HTML(f"""
<div style='border:2px solid #1976D2; border-radius:12px; padding:24px; margin:16px 0; background:white;'>
    <h2 style='margin-top:0; color:#1976D2;'>📋 Consolidated Financial Report</h2>
    <hr style='border:1px solid #eee;'>
    <div style='white-space:pre-wrap; line-height:1.6; font-size:14px;'>{report}</div>
</div>"""))

print(f"\n✅ Report generated — {len(report)} chars")

---
## Section 11 — Multi-Agent Pipeline Test
End-to-end: Summarize → Clarify → Report for one document.

In [ ]:
# Pick the first document for the pipeline test
pipeline_doc = list(doc_contexts.keys())[0]
file_name = doc_contexts[pipeline_doc]["file"]
print(f"🔄 Running 3-agent pipeline on: {file_name}\n")

# Step 1: Summarize
print("Step 1/3: Summarizer...")
t0 = time.time()
step1 = summaries[pipeline_doc]  # reuse
t1 = time.time()

# Step 2: Clarify
print("Step 2/3: Clarification agent...")
step2 = call_agent(AGENTS["clarification"], f"Review this summary and identify gaps or areas needing more detail:\n\n{step1}")
t2 = time.time()

# Step 3: Report
print("Step 3/3: Report generator...")
step3 = call_agent(AGENTS["reporter"], f"""Create a final analyst report incorporating:

Summary:
{step1}

Follow-up areas identified:
{step2}

Generate a polished analyst-ready document.""")
t3 = time.time()

# Pipeline visualization
display(HTML(f"""
<div style='display:flex; align-items:center; gap:8px; margin:20px 0; flex-wrap:wrap;'>
  <div style='background:#2196F3; color:white; padding:12px 20px; border-radius:8px; text-align:center; min-width:140px;'>
    <div style='font-size:20px;'>📄</div>
    <div style='font-weight:bold;'>Summarizer</div>
    <div style='font-size:11px; opacity:0.8;'>{len(step1)} chars</div>
  </div>
  <div style='font-size:24px;'>→</div>
  <div style='background:#FF9800; color:white; padding:12px 20px; border-radius:8px; text-align:center; min-width:140px;'>
    <div style='font-size:20px;'>🔍</div>
    <div style='font-weight:bold;'>Clarification</div>
    <div style='font-size:11px; opacity:0.8;'>{t2-t1:.1f}s</div>
  </div>
  <div style='font-size:24px;'>→</div>
  <div style='background:#4CAF50; color:white; padding:12px 20px; border-radius:8px; text-align:center; min-width:140px;'>
    <div style='font-size:20px;'>📋</div>
    <div style='font-weight:bold;'>Report Gen</div>
    <div style='font-size:11px; opacity:0.8;'>{t3-t2:.1f}s</div>
  </div>
</div>
"""))

# Show final report
display(HTML(f"""
<details style='margin:12px 0;'>
  <summary style='cursor:pointer; font-weight:bold; font-size:15px; color:#1976D2;'>📋 View Final Pipeline Report ({len(step3)} chars)</summary>
  <div style='border:1px solid #ddd; border-radius:8px; padding:16px; margin-top:8px; background:#fafafa;'>
    <pre style='white-space:pre-wrap; font-size:13px; line-height:1.5;'>{step3}</pre>
  </div>
</details>"""))

---
## Section 12 — Export & Final Summary

In [ ]:
# Save detailed results to CSV
results_df.to_csv("eval_results.csv", index=False)

# ── Final summary ──
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

doc_breakdown = ""
for doc_file, grp in results_df.groupby("document_file"):
    avg = grp["score"].mean()
    p = grp["match"].sum()
    t = len(grp)
    bar = "█" * int(avg / 10) + "░" * (10 - int(avg / 10))
    doc_breakdown += f"  {doc_file:<45} {bar} {avg:5.1f}  ({p}/{t} pass)\n"

clar_summary = ", ".join(f"{cr['document'].split('_')[0]}: {cr['questions_asked']}q" for cr in clarification_results)

display(HTML(f"""
<div style='border:2px solid #1a1a2e; border-radius:12px; padding:24px; margin:16px 0; background:linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);'>
  <h2 style='margin-top:0;'>🏦 Financial Agent Evaluation — Final Report</h2>
  <p style='color:#666;'>Run: {timestamp} | Test Pack: financial_agent_test_pack</p>
  <hr>
  <h3>📊 Q&A Accuracy (agent-summarizer)</h3>
  <table style='border-collapse:collapse;'>
    <tr><td style='padding:4px 16px;'>Average Score</td><td style='padding:4px 16px; font-weight:bold;'>{avg_score:.1f} / 100</td></tr>
    <tr><td style='padding:4px 16px;'>Match Rate</td><td style='padding:4px 16px; font-weight:bold;'>{match_rate:.1f}%</td></tr>
    <tr><td style='padding:4px 16px;'>Passed / Total</td><td style='padding:4px 16px; font-weight:bold;'>{pass_count} / {len(results_df)}</td></tr>
  </table>

  <h3>📄 Per-Document Breakdown</h3>
  <pre style='font-size:13px; background:white; padding:12px; border-radius:6px;'>{doc_breakdown}</pre>

  <h3>🔍 Clarification Agent</h3>
  <p>{clar_summary}</p>

  <h3>📋 Report Generator</h3>
  <p>Consolidated report: {len(report)} chars generated</p>

  <h3>🔄 Multi-Agent Pipeline</h3>
  <p>Summarize → Clarify → Report: ✅ Complete</p>

  <hr>
  <p style='font-size:12px; color:#888;'>Results saved to: eval_results.csv, eval_results_by_document.png, eval_score_distribution.png</p>
</div>
"""))

print("\n🎉 Evaluation complete! Artifacts saved.")